In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok transformers bitsandbytes accelerate git+https://github.com/openai/whisper.git
!sudo apt-get install -y ffmpeg

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.2 MB/s e

In [ ]:
import os
import shutil
import whisper
import subprocess
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import torch

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # En prod, restreindre aux domaines autorisés
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Charger Whisper
model_whisper = whisper.load_model("tiny", device=device)

# Config quantification 4-bit pour LLaVA
model_id = "llava-hf/llava-1.5-7b-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_id,use_fast=True)
model_llava = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

@app.post("/analyze")
async def analyze_video(file: UploadFile = File(...)):
    base_dir = "/content"
    video_path = os.path.join(base_dir, file.filename)
    frames_dir = os.path.join(base_dir, "frames")

    os.makedirs(frames_dir, exist_ok=True)

    try:
        # Sauvegarder la vidéo
        with open(video_path, "wb") as f:
            f.write(await file.read())

        # Transcription audio avec Whisper
        result = model_whisper.transcribe(video_path)
        audio_text = result.get("text", "").strip()

        # Extraire images toutes les 2 secondes avec ffmpeg
        subprocess.run(
            [
                "ffmpeg", "-i", video_path,
                "-vf", "fps=1/2",
                os.path.join(frames_dir, "frame_%03d.jpg"),
                "-hide_banner", "-loglevel", "error"
            ],
            check=True
        )

        # Liste des images extraites
        extracted_images = sorted(
            os.path.join(frames_dir, f)
            for f in os.listdir(frames_dir) if f.endswith(".jpg")
        )

        # Générer la description uniquement pour la DERNIÈRE image
        last_description = ""
        if extracted_images:
            image = Image.open(extracted_images[-1]).convert("RGB")
            prompt = "<image>"
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
            generated_ids = model_llava.generate(**inputs, max_new_tokens=150)
            last_description = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    finally:
        # Nettoyer fichiers temporaires
        if os.path.exists(video_path):
            os.remove(video_path)
        if os.path.exists(frames_dir):
            shutil.rmtree(frames_dir)

    return JSONResponse({
        "audio_transcription": audio_text,
        "last_image_description": last_description
    })




Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
!ngrok config add-authtoken 2rJDVpZZ5fek227HTZ7UHDOdqOn_6U28D18JPZeuLrNvmRED8

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
import uvicorn

# Tunnel public via ngrok
public_url = ngrok.connect(8000)
print(f" API disponible à : {public_url}")

# Démarrer le serveur FastAPI
!uvicorn api:app --host 0.0.0.0 --port 8000 --reload

🚀 API disponible à : NgrokTunnel: "https://7769-34-73-104-26.ngrok-free.app" -> "http://localhost:8000"
INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [20041] using WatchFiles
2025-06-10 03:00:38.925892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749524438.957398   20043 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749524438.965745   20043 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the defau

Loading checkpoint shards: 100% 3/3 [01:17<00:00, 25.77s/it]
INFO:     Started server process [20043]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.131:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     196.65.158.1